In [1]:
# Install Required Libraries

# NOTE on why this is structured the way it is:
# - Jupyter's kernel ALWAYS runs an asyncio event loop on the main thread
#   (that's what makes top-level `await` work in cells), and ipykernel sets
#   the PROCESS-WIDE asyncio event loop policy to WindowsSelectorEventLoop
#   at startup.
# - Playwright's ASYNC API fails on Windows because that loop is a
#   SelectorEventLoop, which can't spawn subprocesses -> NotImplementedError.
# - Playwright's SYNC API refuses to run at all if it detects it's on a
#   thread that already has a running event loop -> "Sync API inside the
#   asyncio loop" error. Jupyter's main thread always has one.
# - Running the sync API in a separate thread avoids that second error, but
#   Playwright's sync API STILL internally creates its own asyncio event
#   loop inside that thread, and since the event loop POLICY is process-wide
#   (not per-thread), it still inherits ipykernel's Selector policy and still
#   fails with NotImplementedError.
#
# The fix: inside the worker thread, explicitly switch the event loop policy
# to Proactor right before Playwright starts. This only affects loops
# created AFTER this point, so it does not disturb the kernel's already-
# running main-thread loop.

import asyncio
import sys
import concurrent.futures

from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
import pandas as pd


def _thread_entry(func, args, kwargs):
    if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    return func(*args, **kwargs)


def run_in_thread(func, *args, **kwargs):
    """Run a blocking (sync Playwright) function in its own thread, with a
    Proactor event loop policy, away from Jupyter's main loop."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_thread_entry, func, args, kwargs)
        return future.result()

In [2]:
# Configuration

URL = "https://www.timesjobs.com/job-search?keywords=%22python%22&location=&experience=&refreshed=true"

In [3]:
# Optional sanity check: confirms Playwright launches correctly

def _test_playwright():

    with sync_playwright() as p:

        browser = p.chromium.launch(headless=False)
        page = browser.new_page()

        page.goto("https://www.google.com")
        title = page.title()

        browser.close()

    return title


print(run_in_thread(_test_playwright))

Google


In [4]:
def _get_page_html(url):

    with sync_playwright() as p:

        browser = p.chromium.launch(
            headless=False
        )

        page = browser.new_page()

        try:
            page.goto(
                url,
                timeout=120000,
                wait_until="domcontentloaded"
            )

            page.wait_for_timeout(5000)

            # Scroll page to trigger lazy-loaded job cards
            for i in range(5):
                page.mouse.wheel(0, 3000)
                page.wait_for_timeout(1000)

            html = page.content()

        finally:
            # Always close the browser, even if goto/scroll fails
            browser.close()

    return html


def get_page_html(url):
    return run_in_thread(_get_page_html, url)

In [5]:
# Create HTML Parser

def create_soup(html):

    soup = BeautifulSoup(
        html,
        "lxml"
    )

    return soup

In [6]:
# Extract Job Details
#
# NOTE: TimesJobs was redesigned (Next.js/Tailwind). The old selectors
# (li.job-bx, h3.joblist-comp-name, span.srp-skills, span.sim-posted) no
# longer exist anywhere on the page -- that's why 0 jobs were extracted.
# These selectors were reverse-engineered from the site's CURRENT markup.

def extract_jobs(soup):

    jobs = []

    # Each job listing is now a div.srp-card (not li.job-bx anymore)
    job_cards = soup.select("div.srp-card")

    for job in job_cards:

        # Job Title
        title = job.find("h2")
        title = title.get_text(strip=True) if title else None

        # Company Name + Posted Date (both live in the same info line)
        info_div = job.select_one("div.text-xs.text-gray-400")

        company = None
        posted = None
        if info_div:
            spans = info_div.find_all("span")
            if spans:
                company = spans[0].get_text(strip=True)

            full_text = info_div.get_text(" ", strip=True)
            if "Posted on:" in full_text:
                posted = full_text.split("Posted on:")[-1].strip()

        # Job Link (an <a> overlaying the whole card)
        link_tag = job.select_one("a[href]")
        link = link_tag.get("href") if link_tag else None

        # Skills (span.skill-tag; the last one may just say "+N more")
        skill_spans = job.select("span.skill-tag")
        skills_list = [
            s.get_text(strip=True)
            for s in skill_spans
            if "more" not in s.get_text(strip=True).lower()
        ]
        skills = ", ".join(skills_list) if skills_list else None

        jobs.append({
            "Title": title,
            "Company": company,
            "Skills": skills,
            "Posted": posted,
            "Link": link
        })

    return jobs

In [7]:
# Convert Data into DataFrame

def create_dataframe(jobs):

    df = pd.DataFrame(jobs)

    return df

In [8]:
# Save CSV File

def save_csv(df):

    df.to_csv(
        "timesjobs_python_jobs.csv",
        index=False
    )

    print("CSV file created successfully")

In [9]:
def main():

    print("Starting scraper...")

    html = get_page_html(URL)
    print("HTML downloaded")

    soup = create_soup(html)
    print("HTML parsed")

    jobs = extract_jobs(soup)
    print(f"{len(jobs)} jobs extracted")

    if len(jobs) == 0:
        print(
            "WARNING: 0 jobs found. TimesJobs may have changed its HTML "
            "structure/class names, or the page didn't fully load. "
            "Inspect the page HTML and update the selectors in extract_jobs()."
        )

    df = create_dataframe(jobs)

    save_csv(df)


main()

Starting scraper...
HTML downloaded
HTML parsed
10 jobs extracted
CSV file created successfully
